# RQ4 : Integrated Spatial Diagnostic
## Growth-Utilisation-Risk Overlay | Dholera SIR (2025)

---

> *"Early-stage corridor development in Dholera reveals spatial divergence between where infrastructure is built, where economic activity occurs, and where environmental risk is concentrated."*

| Hypothesis | Statement |
|---|---|
| H1 | Areas with low economic utilisation disproportionately coincide with higher environmental exposure |

**Three-axis classification:**

| Class | Built-up | Economic Activity | Environmental Exposure | Interpretation |
|---|---|---|---|---|
| 🟥 High Misalignment | ✔ New growth | Low (VIIRS dark) | High | Capital deployed in vulnerable, underutilised land |
| 🟨 Transitional | ✔ Under-construction or moderate | Moderate | Moderate | Uncertain trajectory |
| 🟩 Aligned | ✔ Active industrial | High | Low | Spatial alignment across all three axes |

---
## Setup

In [ ]:
import geemap
import geemap.colormaps as cm
import ee
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
geemap.ee_initialize()

---
## Stage 1: Rebuild All Input Layers

RQ4 is a standalone notebook. All layers from RQ1, RQ2, RQ3 are rebuilt here from scratch so this notebook runs independently.

| Layer | Source RQ | Variable |
|---|---|---|
| ROI boundary | — | `roi` |
| Sentinel-2 2025 composite | RQ1 | `s2_2025` |
| Built-up mask 2025 | RQ1 | `built_up_2025` |
| New growth footprint (Class 2) | RQ1 | `new_growth_mask` |
| VIIRS nighttime lights | RQ2 | `nightlight_roi` |
| Cluster typology map | RQ2 | `cluster_map` |
| Environmental Exposure surface | RQ3 | `env_exposure` |

In [ ]:
# ── ROI ──────────────────────────────────────────────────────
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')
roi = geemap.geojson_to_ee(file_path)
print("ROI loaded.")

In [ ]:
# ── Sentinel-2 2025 (Oct–Dec post-monsoon composite) ─────────
s2_2025 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

# ── Sentinel-2 2016 (Oct–Dec post-monsoon composite) ─────────
s2_2016 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2016-10-01', '2016-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

print("Sentinel-2 composites loaded.")

In [ ]:
# ── Spectral Indices ─────────────────────────────────────────
def calculate_indices(image):
    ndbi = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': image.select('B11'), 'NIR': image.select('B8')}
    ).rename('NDBI')
    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)',
        {'Green': image.select('B3'), 'SWIR1': image.select('B11')}
    ).rename('MNDWI')
    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)',
        {'NIR': image.select('B8'), 'Red': image.select('B4')}
    ).rename('SAVI')
    return image.addBands([ndbi, mndwi, savi])

processed_2025 = calculate_indices(s2_2025)
processed_2016 = calculate_indices(s2_2016)
print("Spectral indices computed.")

In [ ]:
# ── Built-up Masks (RQ1 thresholds) ──────────────────────────
built_up_2025 = processed_2025.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)',
    {'NDBI': processed_2025.select('NDBI'),
     'MNDWI': processed_2025.select('MNDWI'),
     'SAVI': processed_2025.select('SAVI')}
).rename('BuiltUp_2025')

built_up_2016 = processed_2016.expression(
    '(NDBI > 0.13) && (MNDWI < 0) && (SAVI < 0.18)',
    {'NDBI': processed_2016.select('NDBI'),
     'MNDWI': processed_2016.select('MNDWI'),
     'SAVI': processed_2016.select('SAVI')}
).rename('BuiltUp_2016')

# ── Growth Classification (RQ1 heatmap) ──────────────────────
change_stack = built_up_2016.rename('y2016').addBands(built_up_2025.rename('y2025'))
growth_class = change_stack.expression(
    "(b('y2016') == 0 && b('y2025') == 0) ? 0"
    ": (b('y2016') == 1 && b('y2025') == 1) ? 1"
    ": (b('y2016') == 0 && b('y2025') == 1) ? 2"
    ": (b('y2016') == 1 && b('y2025') == 0) ? 3"
    ": 0"
).rename('GrowthClass').clip(roi)

# New growth footprint (Class 2)
new_growth_mask = growth_class.eq(2)

print("Built-up masks and growth classification ready.")

In [ ]:
# ── VIIRS Nighttime Lights (RQ2) ──────────────────────────────
# Oct 2025–Mar 2026 median — same window as RQ2
nightlight_roi = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterDate('2025-10-01', '2026-03-31') \
    .select('avg_rad') \
    .median() \
    .clip(roi)

print("VIIRS nighttime lights loaded.")

In [ ]:
# ── Cluster Typology Map (RQ2) ────────────────────────────────
# Recomputed from RQ2 logic for standalone use
ndbi  = s2_2025.normalizedDifference(['B11', 'B8']).rename('NDBI')
mndwi = s2_2025.normalizedDifference(['B3',  'B11']).rename('MNDWI')
savi  = s2_2025.expression(
    '((NIR - RED) * 1.5) / (NIR + RED + 0.5)',
    {'NIR': s2_2025.select('B8'), 'RED': s2_2025.select('B4')}
).rename('SAVI')

# VIIRS threshold: > 0.6 nW/cm²/sr = economically active
is_active_night = nightlight_roi.gt(0.6)

# Under-construction: disturbed soil signature
is_soil_disturbed = savi.gt(0.18).And(savi.lt(0.3)) \
    .And(ndbi.gt(0.05)) \
    .And(mndwi.lt(-0.4))

# Classification (RQ2 priority order)
# 0 = Background, 1 = Active Industrial, 2 = Dormant/Speculative, 3 = Under-Construction
cluster_map = ee.Image(0) \
    .where(is_soil_disturbed,                                 3) \
    .where(built_up_2025.eq(1).And(is_active_night.Not()),    2) \
    .where(built_up_2025.eq(1).And(is_active_night),          1) \
    .clip(roi) \
    .rename('ClusterType')

print("Cluster typology map ready.")
print("  1 = Active Industrial")
print("  2 = Dormant / Speculative")
print("  3 = Under-Construction")

In [ ]:
# ── Environmental Exposure Layer (RQ3) ────────────────────────
# Rebuilding from RQ3 components: Heat Index + FVI

# --- Heat Index ---
ndbi_raw  = processed_2025.select('NDBI')
savi_raw  = processed_2025.select('SAVI')
ndbi_norm = ndbi_raw.add(0.5).divide(1.0).clamp(0, 1).rename('NDBI_norm')
savi_norm = savi_raw.divide(0.8).clamp(0, 1).rename('SAVI_norm')
heat_index = ndbi_norm.subtract(savi_norm).rename('HeatIndex').clip(roi)
heat_norm  = heat_index.add(0.5).divide(1.2).clamp(0, 1).rename('HeatRisk_norm')

# --- FVI (Flood Vulnerability Index) ---
# Rebuilt from RQ3 SAR + terrain pipeline
# If you have already run RQ3 in this session, you can comment out
# this block and use the fvi variable directly

dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
    .select('DEM').filterBounds(roi).mosaic().clip(roi).rename('Elevation')

# SAR dry season permanent water mask (March 2025)
s1_dry = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi).filterDate('2025-03-01', '2025-03-31') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VV')
s1_dry_smoothed = s1_dry.median() \
    .focal_mean(radius=1, kernelType='square', units='pixels').clip(roi)
sar_permanent_water = s1_dry_smoothed.lt(-17).rename('PermanentWater')

# SAR monsoon collection
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi).filterDate('2025-06-15', '2025-10-30') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VV')

def process_sar_image(image):
    smoothed = image.focal_mean(radius=1, kernelType='square', units='pixels')
    water_binary = smoothed.lt(-17).rename('Water')
    monsoon_water = water_binary.where(sar_permanent_water.eq(1), 0)
    return monsoon_water.clip(roi).copyProperties(image, ['system:time_start'])

flood_freq = s1_collection.map(process_sar_image).mean().rename('FloodFreq').clip(roi)

# FVI classification
elev = dem.select('Elevation')
freq = flood_freq.select('FloodFreq')
fvi = ee.Image(1) \
    .where(elev.lte(8).And(freq.gt(0.10)), 2) \
    .where(elev.lte(5).And(freq.gt(0.35)), 3) \
    .rename('FVI').clip(roi) \
    .updateMask(sar_permanent_water.eq(0))

fvi_norm = fvi.subtract(1).divide(2).clamp(0, 1).rename('FloodRisk_norm')

# --- Composite Environmental Exposure ---
env_exposure = heat_norm.multiply(0.5) \
    .add(fvi_norm.multiply(0.5)) \
    .rename('EnvExposure') \
    .clip(roi)

print("Environmental Exposure layer rebuilt.")
print("  Range: 0 (safe) → 1 (maximum heat + flood exposure)")

---
## Stage 2 : Integrated Classification

### Classification Logic

Three binary signals are derived first, then combined:

| Signal | Derivation | Threshold |
|---|---|---|
| `is_builtup` | `built_up_2025 = 1` | Any confirmed built-up surface |
| `is_economically_active` | VIIRS > 0.6 OR cluster = 1 (Active Industrial) | Confirmed nighttime activity |
| `is_high_exposure` | `env_exposure > 0.5` | Above the neutral midpoint of the 0–1 scale |

**Priority order (`.where()`):**
1. Base = Background (0)
2. Aligned (3) — active + low risk
3. Transitional (2) — moderate conditions
4. High Misalignment (1) — built-up + dark + high exposure ← written last, highest priority

In [ ]:
# ── Stage 2: Integrated Spatial Classification ───────────────

# ── Binary signals ────────────────────────────────────────────
# Any confirmed built-up surface (stable or new growth)
is_builtup = built_up_2025.eq(1)

# Economically active: nightlight confirmed OR cluster typology = Active Industrial
is_economically_active = nightlight_roi.gt(0.6).Or(cluster_map.eq(1))

# Under-construction: cluster typology = 3
is_under_construction = cluster_map.eq(3)

# High environmental exposure: above midpoint of composite risk surface
is_high_exposure     = env_exposure.gt(0.5)
is_moderate_exposure = env_exposure.gt(0.25).And(env_exposure.lte(0.5))

# ── Classification conditions ─────────────────────────────────

# 🟩 Aligned Development Zone
# Built-up AND economically active AND NOT high environmental exposure
aligned_cond = is_builtup \
    .And(is_economically_active) \
    .And(is_high_exposure.Not())

# 🟨 Transitional Zone
# Built-up AND (under-construction OR moderate activity)
# AND moderate or high environmental exposure
# — pixels not already classified as Aligned
transitional_cond = is_builtup \
    .And(is_under_construction.Or(is_economically_active)) \
    .And(is_moderate_exposure.Or(is_high_exposure)) \
    .And(aligned_cond.Not())

# 🟥 High Misalignment Zone
# Built-up AND economically inactive AND high environmental exposure
# — written last (highest priority, overwrites Transitional)
misaligned_cond = is_builtup \
    .And(is_economically_active.Not()) \
    .And(is_high_exposure)

# ── Build classification raster ───────────────────────────────
# 0 = Background
# 1 = 🟥 High Misalignment
# 2 = 🟨 Transitional
# 3 = 🟩 Aligned

integrated_class = ee.Image(0) \
    .where(aligned_cond,      3) \
    .where(transitional_cond, 2) \
    .where(misaligned_cond,   1) \
    .rename('IntegratedClass') \
    .clip(roi)

print("Integrated Spatial Classification ready.")
print("  0 = Background (non built-up)")
print("  1 = 🟥 High Misalignment")
print("  2 = 🟨 Transitional")
print("  3 = 🟩 Aligned")

---
## Stage 3 — Integrated Classification Map
### Output 5 (Most Important)

In [ ]:
# ── Output 5: Integrated Classification Map ───────────────────
Map_Integrated = geemap.Map()
Map_Integrated.centerObject(roi, 11)

# Reference
Map_Integrated.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# ── Primary: Integrated Classification ───────────────────────
# Palette: black (background) → red (misaligned) → amber (transitional) → green (aligned)
Map_Integrated.addLayer(
    integrated_class,
    {'min': 0, 'max': 3,
     'palette': ['1a1a1a', 'c1121f', 'f4a261', '2d6a4f']},
    '🗺️ Integrated Spatial Diagnostic'
)

# ── Diagnostic input layers (all off by default) ──────────────
Map_Integrated.addLayer(
    nightlight_roi,
    {'min': 0, 'max': 20,
     'palette': ['000000', 'ffff99']},
    '💡 VIIRS Nightlights (Economic Activity)', shown=False
)

Map_Integrated.addLayer(
    env_exposure,
    {'min': 0, 'max': 1,
     'palette': ['1a1a2e', '0f3460', '533483', 'e94560', 'f5a623', 'ffffff']},
    '⚠️ Environmental Exposure (Heat + Flood)', shown=False
)

Map_Integrated.addLayer(
    cluster_map,
    {'min': 0, 'max': 3,
     'palette': ['1a1a1a', 'e63946', '6a0572', 'f4a261']},
    '🏭 RQ2 Cluster Typology (ref)', shown=False
)

Map_Integrated.addLayer(
    new_growth_mask.selfMask(),
    {'min': 1, 'max': 1, 'palette': ['ffffff']},
    '⬜ New Growth Footprint (2016→2025)', shown=False
)

# ── Legend ────────────────────────────────────────────────────
Map_Integrated.add_legend(
    title='Integrated Spatial Diagnostic',
    legend_dict={
        '⬛ Background (Non Builtup)':                         '1a1a1a',
        '🟥 High Misalignment: Built-up, Dark, High Risk':     'c1121f',
        '🟨 Transitional: Under-construction or Moderate':     'f4a261',
        '🟩 Aligned: Active Industrial, Low Risk':             '2d6a4f',
    },
    position='bottomleft'
)

Map_Integrated.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Output 5: Integrated Classification Map ready.")
print("Toggle diagnostic layers to interrogate each classification axis.")
display(Map_Integrated)

---
## Stage 4 — Area Statistics Table
### Output 6

In [ ]:
# ── Output 6: Area Statistics ─────────────────────────────────
pixel_area = ee.Image.pixelArea()

def class_area_km2(class_val):
    """Total area of a given integrated class across the ROI."""
    mask = integrated_class.eq(class_val).rename('IC')
    result = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    )
    return round(result.getInfo().get('IC', 0) / 1e6, 3)

# ── Compute ───────────────────────────────────────────────────
misaligned_km2   = class_area_km2(1)
transitional_km2 = class_area_km2(2)
aligned_km2      = class_area_km2(3)
total_builtup_km2 = round(misaligned_km2 + transitional_km2 + aligned_km2, 3)



# ── Report ────────────────────────────────────────────────────
print("=" * 65)
print("  OUTPUT 6 — INTEGRATED SPATIAL DIAGNOSTIC: AREA STATISTICS")
print("=" * 65)
print(f"  {'Class':<30} {'Area (km²)':>10}  {'% of Built-up':>14}")
print("-" * 65)
print(f"  {'🟥 High Misalignment':<30} {misaligned_km2:>10.3f}  "
      f"{misaligned_km2/total_builtup_km2*100:>13.1f}%")
print(f"  {'🟨 Transitional':<30} {transitional_km2:>10.3f}  "
      f"{transitional_km2/total_builtup_km2*100:>13.1f}%")
print(f"  {'🟩 Aligned':<30} {aligned_km2:>10.3f}  "
      f"{aligned_km2/total_builtup_km2*100:>13.1f}%")
print("-" * 65)
print(f"  {'Total Built-up':<30} {total_builtup_km2:>10.3f}  {'100.0%':>14}")
print()


---
## Stage 5 — Headline Metric & Final Insight
### Output 7

In [ ]:
# ── Output 7: Headline Metric ─────────────────────────────────
# % of total built-up in High Misalignment zones
pct_misaligned_of_builtup = round(misaligned_km2 / total_builtup_km2 * 100, 1)
pct_misaligned_of_growth  = round(ng_misaligned  / total_growth_km2  * 100, 1)

print("=" * 65)
print("  OUTPUT 7 — HEADLINE METRIC")
print("=" * 65)
print()
print(f"  ★  {pct_misaligned_of_builtup}% of Dholera's total built-up footprint")
print(f"     sits in High Misalignment zones.")
print()
print(f"  ★  {pct_misaligned_of_growth}% of post-2016 new growth specifically")
print(f"     landed in High Misalignment zones.")
print()
print("-" * 65)
print("  H3 TEST RESULT:")
print("  Areas with low economic utilisation disproportionately")
print("  coincide with higher environmental exposure.")
print()

# H3 verdict based on result
if pct_misaligned_of_builtup >= 25:
    print("  ✅ H3 CONFIRMED — A substantial share of built-up land is")
    print("     simultaneously economically dark and environmentally exposed.")
elif pct_misaligned_of_builtup >= 10:
    print("  ⚠️  H3 PARTIALLY SUPPORTED — Misalignment is present but")
    print("     concentrated in specific zones rather than systemic.")
else:
    print("  ❌ H3 WEAK — Limited spatial overlap between economic")
    print("     inactivity and environmental exposure.")

print()
print("-" * 65)
print()
print("  ONE-LINE INSIGHT:")
print("  'Early-stage corridor development in Dholera reveals spatial")
print("   divergence between where infrastructure is built, where economic")
print("   activity occurs, and where environmental risk is concentrated.'")
print("=" * 65)